In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
import gradio as gr 

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

In [ ]:
openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [ ]:
system_message = "you are a helpful assistant"

def message_gpt(prompt):
    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
    response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
    return response.choices[0].message.content

In [ ]:
# This can reveal the "training cut off", or the most recent date in the training data
message_gpt("What is today's date?")

In [ ]:
# User interface using Gradio
def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()

In [ ]:
shout("hello")

In [ ]:
# Here the function shout is passed to the interface as a callback function. Callbacks are functions that are called when a user interacts with the interface.
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch()

In [ ]:
# Share the interface. It will be available at a public URL. It uses HTTP tunneling to make the interface accessible from the outside world.
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(share=True)

In [ ]:
# Open the interface in the browser
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True)

In [ ]:
# Adding authentication to the interface
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True, auth=("raghav", "raghav2004"))

In [ ]:
# Force dark mode. It is done by adding a small JavaScript code to the interface.
force_dark_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never", js=force_dark_mode).launch()

In [ ]:
message_input = gr.Textbox(label="your message:", lines=7)
message_output = gr.Textbox(label="response:", lines=7)

view = gr.Interface(
    fn=shout,
    title="Shout", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=["hello", "howdy"], 
    flagging_mode="never"
)
view.launch()

In [ ]:
# Changing the function to message_gpt
message_input = gr.Textbox(label="your message:", lines=7)
message_output = gr.Textbox(label="response:", lines=7)

view = gr.Interface(
    fn=message_gpt,
    title="GPT", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=["hello, how are you?", "what is the capital of the moon?"], 
    flagging_mode="never"
)
view.launch()

In [ ]:
# Streaming the response
def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
# Generating a response in markdown format
message_input = gr.Textbox(label="your message:", lines=7)
message_output = gr.Markdown(label="response:")

view = gr.Interface(
    fn=stream_gpt,
    title="GPT", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "explain the transformer architecture to a layperson",
        "explain the transformer architecture to an aspiring AI engineer",
    ], 
    flagging_mode="never"
)
view.launch()

In [ ]:
def stream_ollama(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = ollama.chat.completions.create(
        model="qwen3:8b",
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
# Stream the response from a particular model
def stream_model(prompt, model):
    if model=="GPT":
        result = stream_gpt(prompt)
    elif model=="Ollama":
        result = stream_ollama(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [ ]:
message_input = gr.Textbox(label="your message:", info="enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["GPT", "Ollama"], label="select model", value="GPT")
message_output = gr.Markdown(label="response:")

view = gr.Interface(
    fn=stream_model,
    title="LLMs", 
    inputs=[message_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["explain the transformer architecture to a layperson", "GPT"],
            ["explain the transformer architecture to an aspiring AI engineer", "Ollama"]
        ], 
    flagging_mode="never"
    )
view.launch()

In [ ]:
# brochure generation
from scraper import fetch_website_contents

In [ ]:
system_message = """
you are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
respond in markdown without code blocks.
"""

In [ ]:
def stream_brochure(company_name, url, model):
    yield ""
    prompt = f"please generate a company brochure for {company_name}. here is their landing page:\n"
    prompt += fetch_website_contents(url)
    if model=="GPT":
        result = stream_gpt(prompt)
    elif model=="Ollama":
        result = stream_ollama(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [ ]:
name_input = gr.Textbox(label="company name:")
url_input = gr.Textbox(label="landing page URL including http:// or https://")
model_selector = gr.Dropdown(["GPT", "Ollama"], label="select model", value="GPT")
message_output = gr.Markdown(label="response:")

view = gr.Interface(
    fn=stream_brochure,
    title="brochure generator", 
    inputs=[name_input, url_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["hugging face", "https://huggingface.co", "GPT"],
            ["Edward Donner", "https://edwarddonner.com", "Ollama"]
        ], 
    flagging_mode="never"
    )
view.launch()